In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import googlemaps
import time

In [2]:
# Load your Excel file
clinics_df = pd.read_excel('clinics/clinics.xlsx')
clinics_df.head()

,Clinic,City,Country
0,CLINICA COLOMBIA,CALI,COLOMBIA
1,UNIDAD ATENCION PRIMARIA SANITAS TEQUENDAMA,CALI,COLOMBIA
2,CLINICA DESA SAS CALI,CALI,COLOMBIA
3,VIVA 1A IPS CALLE 7,CALI,COLOMBIA
4,SERVIMEDIC QUIRON SAS,CALI,COLOMBIA


In [ ]:
# Set up Google Maps API Client
API_KEY = 'YOUR_GOOGLE_MAPS_API_KEY'
gmaps = googlemaps.Client(key=API_KEY)

# Function to find clinic details using Places API
def get_clinic_details(name, city, country):
    query = f"{name}, {city}, {country}"
    try:
        # Use Places API Text Search to get first match
        results = gmaps.places(query=query)['results']
        if results:
            top_result = results[0]  # taking the first suggestion from Google
            address = top_result.get('formatted_address', None)
            location = top_result['geometry']['location']
            latitude, longitude = location['lat'], location['lng']
            return address, latitude, longitude
        else:
            return None, None, None
    except Exception as e:
        print(f"Error for {query}: {e}")
        return None, None, None

In [4]:
clinics_df = clinics_df[0:10]  # Limit to first 10 rows for testing

In [5]:
# Apply function to dataframe
addresses = []
latitudes = []
longitudes = []

for idx, row in clinics_df.iterrows():
    address, lat, lng = get_clinic_details(row['Clinic'], row['City'], row['Country'])
    addresses.append(address)
    latitudes.append(lat)
    longitudes.append(lng)
    print(f"Fetched {row['Clinic']} -> {address}, Coordinates: {lat}, {lng}")
    time.sleep(0.2)  # to respect rate limits

clinics_df['Address'] = addresses
clinics_df['Latitude'] = latitudes
clinics_df['Longitude'] = longitudes

# Save results
#clinics_df.to_excel('clinics_with_google_details.xlsx', index=False)

#print(clinics_df)

Fetched CLINICA COLOMBIA -> Cra. 46 #9c-85, Cali, Valle del Cauca, Colombia, Coordinates: 3.4148383, -76.5380387
Fetched UNIDAD ATENCION PRIMARIA SANITAS TEQUENDAMA -> Cra. 43a #5c-94, Cali, Valle del Cauca, Colombia, Coordinates: 3.4185642, -76.5443536
Fetched CLINICA DESA SAS CALI -> Cl. 5d #38a 35, San Fernando, Cali, Valle del Cauca, Colombia, Coordinates: 3.42331, -76.5424226
Fetched VIVA 1A IPS CALLE 7 -> Cra. 7 #41-34, Los Cambulos, Cali, Valle del Cauca, Colombia, Coordinates: 3.4200636, -76.5414938
Fetched SERVIMEDIC QUIRON SAS -> Cra. 64a #12A-149, Comuna 17, Cali, Valle del Cauca, Colombia, Coordinates: 3.4014703, -76.53705049999999
Fetched UPSS CAMU LA GRANJA - EL AMPARO -> Av 1 Cr 1 Manzana E, lote 1, Simón, Bolivar, Colombia, Coordinates: 8.7462701, -75.89816549999999
Fetched CLINICA COLSANITAS SA SEBASTIAN DE BELALCAZAR -> Av. 4 Nte. #7N-81, Granada, Cali, Valle del Cauca, Colombia, Coordinates: 3.454688, -76.53713119999999
Fetched FUNDACION VALLE DEL LILI -> Cra. 98 #18

In [ ]:

# Initialize Nominatim Geocoder
geolocator = Nominatim(user_agent="clinic_locator")

# RateLimiter ensures you don't exceed usage limits
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

In [ ]:
# Create an "Address" column by combining city and country
clinics_df['Address'] = clinics_df['City'] + ', ' + clinics_df['Country']

In [ ]:
# Get coordinates
clinics_df['Coordinates'] = clinics_df['Address'].apply(lambda addr: geocode(addr))

In [ ]:

# Extract latitude and longitude separately
clinics_df['Latitude'] = clinics_df['Coordinates'].apply(lambda loc: loc.latitude if loc else None)
clinics_df['Longitude'] = clinics_df['Coordinates'].apply(lambda loc: loc.longitude if loc else None)

# Drop intermediate columns if desired
clinics_df.drop(columns=['Coordinates'], inplace=True)

# Save results to a new Excel file
clinics_df.to_excel('clinics/clinics_with_coordinates.xlsx', index=False)

print(clinics_df)
